# exp133_gr_bimodal_match_ambiguity_detector train

Target-free GR score-curve ambiguity detector for +/-15-25ft decoys and multi-peak matching risk. The notebook saves diagnostic tables and a downstream feature cache; it does not train LightGBM.


## Contents

1. Setup and configuration
2. Input and diagnostic contract
3. Run GR bimodal ambiguity detector
4. Preview outputs
5. Metrics and next branch


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from gr_bimodal_match_ambiguity_detector import run_train_from_config, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

if DEBUG:
    config.setdefault("audit", {})["max_prediction_rows"] = int(os.environ.get("EXPERIMENT_MAX_ROWS", "20000"))
    config.setdefault("audit", {})["max_feature_rows"] = int(os.environ.get("EXPERIMENT_MAX_ROWS", "20000"))

print(json.dumps({
    "experiment": EXPERIMENT_NAME,
    "route": get_nested(config, "experiment.route"),
    "status": get_nested(config, "experiment.status"),
    "parent": get_nested(config, "lineage.parent"),
    "cache_parent": get_nested(config, "lineage.cache_parent"),
    "mode": get_nested(config, "audit.mode"),
    "debug": DEBUG,
    "max_prediction_rows": get_nested(config, "audit.max_prediction_rows"),
    "max_feature_rows": get_nested(config, "audit.max_feature_rows"),
    "artifacts_dir": str(paths.artifacts_dir),
}, indent=2, sort_keys=True))


## 2. Input and diagnostic contract


In [ ]:
print(json.dumps({
    "gr_bimodal_ambiguity": get_nested(config, "model.gr_bimodal_ambiguity"),
    "leakage_policy": get_nested(config, "validation.leakage_policy"),
    "expected_train_artifacts": get_nested(config, "audit.expected_train_artifacts"),
}, indent=2, ensure_ascii=False))

for label, dotted in {
    "train_dir": "data.train_dir",
    "exp072_feature_cache": "data.exp072_feature_cache",
    "exp073_predictions": "data.exp073_predictions",
    "exp092_predictions": "data.exp092_predictions",
}.items():
    print(label, json.dumps(get_nested(config, dotted), ensure_ascii=False))


## 3. Run GR bimodal ambiguity detector


In [ ]:
summary = run_train_from_config(config, output_dir=paths.artifacts_dir)
print(json.dumps(to_jsonable({
    "status": summary["status"],
    "runtime_seconds": summary["runtime_seconds"],
    "rows": summary["rows"],
    "wells": summary["wells"],
    "ambiguity": summary["ambiguity"],
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
    "feature_cache": {
        "variant": summary["feature_cache"]["variant"],
        "feature_count": summary["feature_cache"]["feature_count"],
        "outputs": summary["feature_cache"]["outputs"],
    },
    "recommendation": summary["recommendation"],
}), indent=2, sort_keys=True))


## 4. Preview outputs


In [ ]:
artifact_paths = {name: paths.artifacts_dir / filename for name, filename in summary["outputs"].items() if filename}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

def preview_csv(name: str, n: int = 10) -> pd.DataFrame:
    path = artifact_paths[name]
    frame = pd.read_csv(path, nrows=n)
    display(frame)
    return frame

candidate_metrics_preview = preview_csv("candidate_metrics")
bucket_metrics_preview = preview_csv("bucket_metrics")
well_metrics_preview = preview_csv("well_metrics")
well_input_preview = preview_csv("well_input_summary")


## 5. Metrics and next branch


In [ ]:
metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": summary["status"],
    "updated_at": datetime.now(UTC).isoformat(),
    "route": get_nested(config, "experiment.route"),
    "metric": "gr_bimodal_match_ambiguity_diagnostic",
    "rows": summary["rows"],
    "wells": summary["wells"],
    "ambiguity": summary["ambiguity"],
    "best_candidate_by_rmse": summary["best_candidate_by_rmse"],
    "feature_cache": summary["feature_cache"],
    "source": summary["source"],
    "outputs": summary["outputs"],
    "recommendation": summary["recommendation"],
}
metrics_path = paths.experiment_dir / "metrics.json"
with metrics_path.open("w") as fp:
    json.dump(to_jsonable(metrics), fp, indent=2, sort_keys=True)
print(json.dumps(to_jsonable(metrics), indent=2, sort_keys=True))
